In [ ]:
# Clone the repository
!git clone https://github.com/huggingface/lerobot.git

# Move into the folder
%cd lerobot

# Install LeRobot with SmolVLA dependencies
!pip install -e ".[smolvla]"

# Install server tools AND nest-asyncio for Colab compatibility
!pip install fastapi uvicorn pyngrok pydantic nest-asyncio

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Authenticate your ngrok tunnel (Requires adding 'ngrok_auth_token' to Colab Secrets)
ngrok.set_auth_token(userdata.get('ngrok_auth_token'))

# Open a tunnel on port 8000
public_url = ngrok.connect(8000).public_url
print(f"--> SUCCESS! NGROK PUBLIC URL: {public_url}")

In [ ]:
import sys
import os
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import torch
import base64
import numpy as np
from io import BytesIO
from PIL import Image

# 1. Apply the async patch so Uvicorn can run inside Colab
nest_asyncio.apply()

# 2. Hardcode the path so Python finds the cloned lerobot library
sys.path.insert(0, "/content/lerobot/src")
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

app = FastAPI()

# 3. Target the T4 GPU (or fallback to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Server] Loading SmolVLA on {device}...")

# ============================================================
# 4. Load YOUR fine‑tuned model from Hugging Face
# ============================================================
model_path = "leumals/smolvla_ur5e_finetuned"
policy = SmolVLAPolicy.from_pretrained(
    model_path,
    torch_dtype=torch.float32
)

# 5. Update model configuration to match your training settings
#    (these are the values used during training)
policy.config.num_steps = 30           # inference steps (your training used 30)
policy.config.chunk_size = 50          # action chunk size (matches config.json)
policy.config.n_action_steps = 50      # number of action steps (matches config.json)

policy.use_cache = False

policy.to(device)
policy.eval()

class InferenceRequest(BaseModel):
    image_top: str          # observation.images.image
    image_depth: str        # observation.images.image_with_depth
    image_hand: str         # observation.images.hand_image
    instruction: str

@app.post("/predict")
def predict(request: InferenceRequest):
    try:
        # Decode base64 images
        top_img_data = base64.b64decode(request.image_top)
        top_image = Image.open(BytesIO(top_img_data)).convert("RGB")

        depth_img_data = base64.b64decode(request.image_depth)
        depth_image = Image.open(BytesIO(depth_img_data)).convert("RGB")

        hand_img_data = base64.b64decode(request.image_hand)
        hand_image = Image.open(BytesIO(hand_img_data)).convert("RGB")

        # Resize images to 480x640 (your training shape)
        # The model will then resize them internally to 512x512 for processing.
        target_size = (640, 480)  # (width, height)
        top_image = top_image.resize(target_size)
        depth_image = depth_image.resize(target_size)
        hand_image = hand_image.resize(target_size)

        # Convert to Tensor [3, H, W] and normalize to [0,1]
        top_tensor = torch.from_numpy(np.array(top_image)).permute(2, 0, 1).float() / 255.0
        depth_tensor = torch.from_numpy(np.array(depth_image)).permute(2, 0, 1).float() / 255.0
        hand_tensor = torch.from_numpy(np.array(hand_image)).permute(2, 0, 1).float() / 255.0

        # Tokenize instruction
        tokenized_text = policy.model.vlm_with_expert.processor.tokenizer(
            request.instruction,
            return_tensors="pt",
            padding=True
        )

        # Build observation dictionary WITH YOUR MODEL'S EXPECTED KEYS
        observation = {
            "observation.images.image": top_tensor.unsqueeze(0).to(device),
            "observation.images.image_with_depth": depth_tensor.unsqueeze(0).to(device),
            "observation.images.hand_image": hand_tensor.unsqueeze(0).to(device),
            "observation.state": torch.zeros((1, 8)).to(device),   # 8‑dim state as in your config
            "observation.language.tokens": tokenized_text.input_ids.to(device),
            "observation.language.attention_mask": tokenized_text.attention_mask.to(device).bool()
        }

        # Inference
        with torch.no_grad():
            action_chunk = policy.select_action(observation)

        # Return all predicted actions (chunk of 50 actions, each 7‑dimensional)
        actions = action_chunk.cpu().numpy().tolist()
        return {"status": "success", "actions": actions}

    except Exception as e:
        import traceback
        print(f" -> [Server Side Error] {str(e)}")
        traceback.print_exc()
        return {"status": "error", "message": str(e)}

# 6. Start the server in a background thread
import threading
import asyncio

def run_server():
    # Create a brand new event loop specifically for this thread
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

print("Starting FastAPI server in the background...")
threading.Thread(target=run_server, daemon=True).start()
print("Server is live 🎉 You can now send requests from Webots.")